# Spotify Listening Analysis: workbook
**Question:** How does this listener use Spotify, what makes them skip, and can a skip be predicted before it happens?

**How to use this workbook**
1. Read the task, then try it yourself using the hints. Google is allowed; that's how analysts actually work.
2. Run it and compare with the ✅ checkpoint.
3. Stuck for more than 10 to 15 minutes? Ask Claude for a *hint* (not the answer), or open `answer_key.ipynb` at the same section.
4. After each section, write a one-line 📝 note in your own words. These become your README and interview answers.

Useful references: pandas docs (pandas.pydata.org), and searching "pandas how to …".

## 0. Setup

Run this cell as it is. These are settings you'll use throughout.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

DATA_PATH = "data/spotify_history.csv"
MIN_STREAM_MS = 30_000     # Spotify counts a play as a "stream" at 30 seconds
MIN_PLAYS_PER_ARTIST = 20  # minimum plays before ranking an artist's skip rate
SESSION_GAP_MIN = 30       # a gap longer than this (minutes) starts a new session

## 1. Load

### 1a. Read the CSV
**Task:** Load the CSV at `DATA_PATH` into a DataFrame called `raw`. Show how many rows and columns it has, and the first 5 rows.

**Hints:** `pd.read_csv()`, `.shape`, `.head()`

**✅ Checkpoint:** Roughly 150,000 rows and 11 columns. The columns should match the data dictionary.

In [ ]:
# your code here

### 1b. Column types
**Task:** Show each column's data type and how many non-null values it has.

**Hints:** `.info()`

**✅ Checkpoint:** `ts` will show as text (object/str), not a date yet. That's something you'll fix in cleaning.

In [ ]:
# your code here

## 2. Explore (before changing anything)

### 2a. Missing values
**Task:** Count missing values in each column, sorted from most to fewest.

**Hints:** `.isna()`, `.sum()`, `.sort_values(ascending=False)`

**✅ Checkpoint:** You should see some columns with missing values. Which ones?

In [ ]:
# your code here

**📝 Note:** Which columns have missing values, and how many? Why might that be?

-

### 2b. What's in the categories?
**Task:** For `platform`, `reason_start`, `reason_end`, `shuffle` and `skipped`, show how often each value appears, including blanks.

**Hints:** `.value_counts(dropna=False)`. Try a `for` loop over a list of column names so you don't repeat yourself.

**✅ Checkpoint:** You'll see codes like `trackdone` and `fwdbtn`. Guess what each means.

In [ ]:
# your code here

**📝 Note:** What do the reason codes mean? Anything odd in `platform`?

-

### 2c. How long are plays?
**Task:** Get summary statistics for `ms_played`. Then work out what % of plays are under 30 seconds.

**Hints:** `.describe()`. For the %: `(raw['ms_played'] < MIN_STREAM_MS).mean()`. A True/False column's mean is the share that are True.

**✅ Checkpoint:** The minimum might be 0, which is worth noting.

In [ ]:
# your code here

## 3. Clean

Work on a copy: `df = raw.copy()`. Keep a list called `clean_log` and append a sentence every time you remove or change something.

### 3a. Start the copy and log
**Task:** Create `df` as a copy of `raw` and an empty list `clean_log`. Save the starting row count as `n0`.

**Hints:** `.copy()`, `len()`

**✅ Checkpoint:** `len(df)` matches `len(raw)`.

In [ ]:
# your code here

### 3b. Drop rows with no track or artist
**Task:** Remove rows where `track_name` or `artist_name` is missing. Log how many you removed.

**Hints:** `.dropna(subset=[...])`. Compare `len(df)` before and after.

**✅ Checkpoint:** A small number of rows are removed.

In [ ]:
# your code here

### 3c. Fix timestamps
**Task:** Convert `ts` to a real datetime.

**Hints:** `pd.to_datetime()`

**✅ Checkpoint:** `df['ts'].dtype` shows `datetime64`. Remember from the dictionary: `ts` is when the track **stopped**.

In [ ]:
# your code here

### 3d. Duplicates
**Task:** Remove exact duplicate rows and log how many there were.

**Hints:** `.drop_duplicates()`

**✅ Checkpoint:** Maybe zero, maybe not. Either way, log it.

In [ ]:
# your code here

### 3e. Make `shuffle` and `skipped` clean True/False
**Task:** Check what type these columns are and whether `skipped` has blanks. Decide what a blank `skipped` means, make both columns proper booleans, and log your decision.

**Hints:** `.dtype`, `.isna().sum()`, `.fillna()`, `.astype(bool)`

**✅ Checkpoint:** `df['skipped'].value_counts(dropna=False)` shows only True and False.

In [ ]:
# your code here

**📝 Note:** What did you decide about blank `skipped` values, and why?

-

### 3f. Sort by time
**Task:** Sort `df` by `ts`, oldest first, and reset the index.

**Hints:** `.sort_values()`, `.reset_index(drop=True)`

**✅ Checkpoint:** `df['ts'].head()` shows the earliest dates.

In [ ]:
# your code here

## 4. Define metrics and add features

### 4a. Stream and skip
**Task:** Add `is_stream` (played at least `MIN_STREAM_MS`) and `is_skip` (`skipped` is True **or** `reason_end` is `'fwdbtn'`).

**Hints:** Comparisons give True/False columns. Use `|` for OR, and wrap each condition in brackets.

**✅ Checkpoint:** `df['is_skip'].mean()` gives your overall skip rate.

In [ ]:
# your code here

**📝 Note:** Why define a skip using two conditions instead of just `skipped`?

-

### 4b. Time features
**Task:** Add `hour`, `weekday` (day name), `month` (e.g. 2023-05) and `minutes_played`.

**Hints:** The `.dt` accessor: `.dt.hour`, `.dt.day_name()`, `.dt.to_period('M')`.

**✅ Checkpoint:** `df[['ts','hour','weekday','month']].head()` looks right.

In [ ]:
# your code here

### 4c. Familiarity features
**Task:** For each play, how many times had this track and this artist been played *before*? Add `track_prior_plays`, `artist_prior_plays` and `artist_is_new` (True on the first ever play of an artist).

**Hints:** `df.groupby('track_name').cumcount()` numbers each play within its group, 0 for the first. It only works correctly because you sorted by time.

**✅ Checkpoint:** The first row has 0 prior plays.

In [ ]:
# your code here

### 4d. Listening sessions
**Task:** Start a new session whenever the gap since the previous play is more than `SESSION_GAP_MIN` minutes. Add `session_id`, and `position_in_session` (0 for the first track of a session).

**Hints:** `df['ts'].diff()` gives gaps. `.dt.total_seconds() / 60` converts them to minutes. A True each time a new session starts, then `.cumsum()`, gives an ID. Then use `groupby('session_id').cumcount()`.

**✅ Checkpoint:** `df['session_id'].nunique()` gives a sensible number of sessions (thousands, not 1 and not 150,000).

In [ ]:
# your code here

In [ ]:
# your code here

### 4e. Save for Tableau
**Task:** Save `df` to `output/cleaned.csv`.

**Hints:** `os.makedirs('output', exist_ok=True)`, `.to_csv(..., index=False)`

**✅ Checkpoint:** The file appears in the `output` folder.

In [ ]:
# your code here

## 5. Analyse

### 5a. Headline numbers
**Task:** Calculate: date range, total plays, total hours, unique artists, unique tracks, skip rate, % of plays under 30s, and number of sessions.

**Hints:** `.min()`, `.max()`, `.sum()`, `.nunique()`, `.mean()`. Put them in a dictionary, then `pd.Series(...)`, for a neat table.

**✅ Checkpoint:** These become your dashboard KPI tiles.

In [ ]:
# your code here

### 5b. Listening heatmap
**Task:** Make a heatmap of hours listened by weekday (rows, Monday first) and hour of day (columns).

**Hints:** `df.pivot_table(index=..., columns=..., values='minutes_played', aggfunc='sum')`, then `.reindex([...days in order])`, then `sns.heatmap()`.

**✅ Checkpoint:** A grid with clear busy and quiet times.

In [ ]:
# your code here

**📝 Note:** When does this person listen most?

-

### 5c. When do they skip?
**Task:** Plot skip rate by hour. Then compare skip rate with shuffle on vs off, and for new vs known artists.

**Hints:** `df.groupby('hour')['is_skip'].mean().plot(kind='bar')`

**✅ Checkpoint:** Look for hours that stand out.

In [ ]:
# your code here

In [ ]:
# your code here

**📝 Note:** Which factor makes the biggest difference to skipping?

-

### 5d. Most and least skipped artists
**Task:** For each artist, get number of plays and skip rate. Keep only artists with at least `MIN_PLAYS_PER_ARTIST` plays, then chart the 10 most and 10 least skipped.

**Hints:** `.groupby('artist_name').agg(plays=('is_skip','size'), skip_rate=('is_skip','mean'))`, then filter, then `.nlargest()` / `.nsmallest()`.

**✅ Checkpoint:** Try it without the 20-play filter first and see why the filter matters.

In [ ]:
# your code here

In [ ]:
# your code here

### 5e. Discovery and retention
**Task:** (1) Plot new artists discovered per month. (2) Of artists first played at least 90 days before the data ends, what % were still being played 90+ days after first listen?

**Hints:** Get each artist's first and last play with `groupby(...)['ts'].min()` / `.max()`. Subtracting dates gives a time difference you can compare with `pd.Timedelta(days=90)`.

**✅ Checkpoint:** One line chart and one percentage.

In [ ]:
# your code here

In [ ]:
# your code here

### 📝 Findings: pick your best 3 to 5
Write each as **insight + so what**.
1. 
2. 
3.

## 6. Model: can a skip be predicted?
Before writing any code, answer this in the note below:
**Which columns would be cheating if used to predict a skip, because they're only known after the track ends?**
This is called *data leakage*, and explaining it well is one of the strongest things you can say in an interview.

**📝 Note:** columns that would leak the answer, and why:

-

### 6a. Build the features and target
**Task:** Create `X` with: hour, track_prior_plays, artist_prior_plays, position_in_session, shuffle, artist_is_new, weekday, platform and reason_start. Turn the text columns into numbers. Create `y` = `is_skip` as 0/1.

**Hints:** `pd.get_dummies(..., columns=[...], drop_first=True)`, `.astype(int)`

**✅ Checkpoint:** `X.shape` gives (rows, roughly 15 to 25 columns).

In [ ]:
# your code here

### 6b. The baseline
**Task:** If a 'model' always guessed the most common outcome, what accuracy would it get?

**Hints:** `y.mean()` is the skip share. The baseline is whichever is bigger: that or 1 minus it.

**✅ Checkpoint:** This number is why accuracy alone is misleading.

In [ ]:
# your code here

**📝 Note:** What's the baseline accuracy, and why does it matter?

-

### 6c. Split by time
**Task:** Use the first 80% of rows to train and the last 20% to test. Don't shuffle them: you're predicting the future from the past.

**Hints:** `split = int(len(X) * 0.8)`, `X.iloc[:split]`, `X.iloc[split:]`

**✅ Checkpoint:** The training set has 4× as many rows as the test set.

In [ ]:
# your code here

**📝 Note:** Why split by time instead of randomly?

-

### 6d. Train two models
**Task:** Train `LogisticRegression` and `RandomForestClassifier` (both with `class_weight='balanced'`). For each, print a classification report and ROC AUC on the test set.

**Hints:** `from sklearn.linear_model import LogisticRegression`, `from sklearn.ensemble import RandomForestClassifier`, `from sklearn.metrics import classification_report, roc_auc_score`. Use `.fit()`, `.predict()`, `.predict_proba()[:, 1]`.

**✅ Checkpoint:** ROC AUC between 0.5 (random guessing) and 1.0 (perfect). Look at precision and recall for 'skipped'.

In [ ]:
# your code here

In [ ]:
# your code here

**📝 Note:** Which model did better, and on which metric? What do precision and recall mean in plain English here?

-

### 6e. What drives skips?
**Task:** Plot the random forest's 12 most important features.

**Hints:** `pd.Series(model.feature_importances_, index=X.columns).nlargest(12).plot(kind='barh')`

**✅ Checkpoint:** A ranked bar chart.

In [ ]:
# your code here

**📝 Note:** Top 3 drivers of skipping, and what would you add with more time?

-

## 7. Cleaning log

### 7. Print the log
**Task:** Print the starting and final row counts and every line in `clean_log`, plus your definitions of a stream, a skip and a session.

**Hints:** A `for` loop over `clean_log`.

**✅ Checkpoint:** Copy this straight into your README's Cleaning section.

In [ ]:
# your code here